# PlayMind | 01. Análisis exploratorio de datos (EDA)
**Proyecto final - Fundamentos de Inteligencia Artificial**  
**Entrega:** lunes 16 de noviembre de 2026 · **Equipo:** Karen Cruces, Romina Garcia, Said Jimenez, Diego Flores

Este cuaderno revisa el comportamiento de jugadores y las imágenes faciales (FER-2013) **sin entrenar modelos todavía**. Incluye distribuciones, correlaciones, valores atípicos, faltantes, conteo por emoción y balance de clases.

## 1. Preparación: rutas y librerías

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from IPython.display import display

sns.set_theme(style="whitegrid")

# Funciona si Jupyter se abrió en la raíz del repo o dentro de notebooks/.
actual = Path.cwd().resolve()
RAIZ = actual.parent if actual.name.lower() == 'notebooks' else actual
DATOS = RAIZ / 'data' / 'raw'
FIGURAS = RAIZ / 'reports' / 'figuras'
FIGURAS.mkdir(parents=True, exist_ok=True)
print('Proyecto:', RAIZ)
print('Carpeta esperada de datos:', DATOS)
print('Carpeta de gráficas:', FIGURAS)

## 2. Cargar datos tabulares
**Dataset:** *Predict Online Gaming Behavior Dataset* (Kaggle). Se carga el CSV ubicado en `data/raw/`.

In [ ]:
csvs = list(DATOS.rglob('*.csv')) if DATOS.exists() else []
candidatos = [p for p in csvs if any(s in p.name.lower() for s in ['gaming', 'player', 'behavior']) and 'fer' not in p.name.lower()]
ruta_csv = sorted(candidatos)[0] if candidatos else None

if ruta_csv is None:
    datos = None
    print('No se encontró el CSV tabular en data/raw/.')
    print('Fuente: https://www.kaggle.com/datasets/rabieelkharoua/predict-online-gaming-behavior-dataset')
    print('Archivos CSV encontrados:', [str(p.relative_to(RAIZ)) for p in csvs])
else:
    try:
        datos = pd.read_csv(ruta_csv)
    except UnicodeDecodeError:
        datos = pd.read_csv(ruta_csv, encoding='latin1')
    print('Archivo cargado:', ruta_csv.relative_to(RAIZ))
    print('Filas:', datos.shape[0], '| Columnas:', datos.shape[1])
    display(datos.head())

## 3. Dimensiones, tipos, duplicados y estadística descriptiva
Este primer vistazo sirve para conocer la cantidad de jugadores, columnas, tipos de variables y posibles inconsistencias.

In [ ]:
if datos is not None:
    print('Dimensiones:', datos.shape)
    print('Columnas y tipos:')
    display(pd.DataFrame({'columna': datos.columns, 'tipo': datos.dtypes.astype(str).values}))
    print('Filas duplicadas:', int(datos.duplicated().sum()))
    display(datos.describe(include='all').T)
else:
    print('Sin análisis tabular: primero agrega el archivo CSV.')

### 3.1 Variables categóricas
Género, ubicación, género de videojuego y dificultad son categorías. Las revisamos aparte porque las correlaciones de Pearson solo aplican a variables numéricas. Más adelante habrá que codificarlas (one-hot) para los modelos.

In [ ]:
if datos is not None:
    categoricas = datos.select_dtypes(exclude=np.number).columns.tolist()
    # InGamePurchases es 0/1: se trata como categórica para esta vista.
    if 'InGamePurchases' in datos.columns and 'InGamePurchases' not in categoricas:
        categoricas.append('InGamePurchases')
    categoricas = [c for c in categoricas if c != 'EngagementLevel']
    if categoricas:
        n = len(categoricas)
        filas_g = (n + 2) // 3
        fig, ejes = plt.subplots(filas_g, 3, figsize=(14, 4*filas_g), squeeze=False)
        for eje, col in zip(ejes.flat, categoricas):
            datos[col].value_counts().plot.bar(ax=eje)
            eje.set_title('Jugadores por ' + col)
            eje.set_ylabel('Jugadores'); eje.set_xlabel('')
            eje.tick_params(axis='x', rotation=30)
        for eje in list(ejes.flat)[n:]: eje.axis('off')
        fig.tight_layout()
        fig.savefig(FIGURAS / '00_categoricas.png', dpi=150)
        plt.show()
        for col in categoricas:
            print(col, '->', datos[col].nunique(), 'categorías')

## 4. Valores faltantes
Calculamos cuántos datos faltan por variable. No se supondrá que están completos hasta ejecutar esta sección.

In [ ]:
if datos is not None:
    faltantes = pd.DataFrame({
        'faltantes': datos.isna().sum(),
        'porcentaje': (datos.isna().mean() * 100).round(2)
    }).sort_values('faltantes', ascending=False)
    display(faltantes)
    fig, ax = plt.subplots(figsize=(10, 4))
    faltantes['porcentaje'].plot.bar(ax=ax)
    ax.set_title('Porcentaje de datos faltantes por columna')
    ax.set_ylabel('Porcentaje (%)'); ax.set_xlabel('Variable')
    fig.tight_layout()
    fig.savefig(FIGURAS / '01_faltantes.png', dpi=150)
    plt.show()
    if faltantes['faltantes'].sum() == 0:
        print('Interpretación: en el CSV cargado no se detectaron valores faltantes.')
    else:
        print('Interpretación: revisar las columnas con faltantes antes del modelado.')

## 5. Distribuciones numéricas
Revisamos variables como horas jugadas, sesiones semanales, nivel del jugador y duración de sesión. Las formas de los histogramas pueden indicar concentración, sesgos o valores extremos.

In [ ]:
if datos is not None:
    preferidas = ['PlayTimeHours', 'SessionsPerWeek', 'PlayerLevel', 'AvgSessionDurationMinutes']
    columnas = [c for c in preferidas if c in datos.columns and pd.api.types.is_numeric_dtype(datos[c])]
    if not columnas:
        columnas = datos.select_dtypes(include=np.number).columns.drop('PlayerID', errors='ignore').tolist()[:4]
    if columnas:
        fig, ejes = plt.subplots(2, 2, figsize=(11, 7))
        for eje, col in zip(ejes.flat, columnas):
            eje.hist(datos[col].dropna(), bins=25)
            eje.set_title('Distribución de ' + col)
            eje.set_xlabel(col)
            eje.set_ylabel('Cantidad de jugadores')
        for eje in list(ejes.flat)[len(columnas):]: eje.axis('off')
        fig.tight_layout()
        fig.savefig(FIGURAS / '02_distribuciones.png', dpi=150)
        plt.show()
        resumen = datos[columnas].describe().T
        resumen['asimetria'] = datos[columnas].skew().round(2)
        display(resumen)
    else:
        print('No se encontraron columnas numéricas para graficar.')

## 6. Correlaciones
Usamos correlación de Pearson para explorar **relaciones lineales entre variables numéricas**. Una correlación no significa causalidad.

In [ ]:
if datos is not None:
    numericas = datos.select_dtypes(include=np.number).drop(columns=['PlayerID'], errors='ignore')
    if numericas.shape[1] >= 2:
        matriz = numericas.corr()
        display(matriz.round(2))
        fig, ax = plt.subplots(figsize=(9, 7))
        mapa = ax.imshow(matriz, cmap='coolwarm', vmin=-1, vmax=1)
        ax.set_xticks(range(len(matriz.columns)), matriz.columns, rotation=60, ha='right')
        ax.set_yticks(range(len(matriz.index)), matriz.index)
        for i in range(len(matriz.index)):
            for j in range(len(matriz.columns)):
                ax.text(j, i, f'{matriz.iloc[i, j]:.2f}', ha='center', va='center', fontsize=8)
        fig.colorbar(mapa, ax=ax, shrink=0.8)
        ax.set_title('Correlaciones entre variables numéricas')
        fig.tight_layout()
        fig.savefig(FIGURAS / '03_correlaciones.png', dpi=150)
        plt.show()
        # Pares más correlacionados (sin la diagonal)
        pares = matriz.where(np.triu(np.ones(matriz.shape), k=1).astype(bool)).stack()
        print('Cinco pares con mayor correlación absoluta:')
        display(pares.reindex(pares.abs().sort_values(ascending=False).index).head(5).round(3).to_frame('correlacion'))
    else:
        print('Se requieren al menos dos columnas numéricas para correlaciones.')

## 7. Valores atípicos (regla IQR)
Un valor atípico es una observación fuera del intervalo Q1 - 1.5 × IQR a Q3 + 1.5 × IQR. **No lo eliminaremos automáticamente**, porque puede representar un jugador real.

In [ ]:
if datos is not None:
    nums = datos.select_dtypes(include=np.number).drop(columns=['PlayerID'], errors='ignore')
    filas = []
    for col in nums.columns:
        serie = nums[col].dropna()
        if serie.empty: continue
        q1, q3 = serie.quantile([0.25, 0.75])
        iqr = q3-q1
        if iqr == 0:
            cantidad = 0  # Revisar por separado columnas con dispersión nula.
        else:
            cantidad = int(((serie < q1-1.5*iqr) | (serie > q3+1.5*iqr)).sum())
        filas.append({'variable':col, 'valores_atipicos':cantidad,
                      'porcentaje':round(100*cantidad/len(serie),2)})
    tabla_atipicos = pd.DataFrame(filas)
    display(tabla_atipicos)
    columnas_box = [c for c in ['PlayTimeHours', 'SessionsPerWeek', 'PlayerLevel'] if c in nums.columns]
    if columnas_box:
        fig, ejes = plt.subplots(1, len(columnas_box), figsize=(4*len(columnas_box), 4), squeeze=False)
        for eje,col in zip(ejes.flat,columnas_box):
            eje.boxplot(datos[col].dropna())
            eje.set_xticks([1], [col])
            eje.set_title('Atípicos en '+col)
        fig.tight_layout()
        fig.savefig(FIGURAS / '04_atipicos.png', dpi=150)
        plt.show()

## 8. Variable objetivo y balance del dataset tabular
Se propone **`EngagementLevel`** con las categorías **Low, Medium y High**, porque este dataset registra interés o participación, **no un abandono comprobado**. No llamaremos "abandono" al objetivo salvo que tengamos etiquetas reales de abandono.

In [ ]:
OBJETIVO = 'EngagementLevel'
ORDEN = ['Low', 'Medium', 'High']
if datos is not None:
    if OBJETIVO in datos.columns:
        conteos = datos[OBJETIVO].value_counts(dropna=False)
        conteos = conteos.reindex([o for o in ORDEN if o in conteos.index] + [o for o in conteos.index if o not in ORDEN])
        display(pd.DataFrame({'jugadores':conteos, 'porcentaje':(conteos/len(datos)*100).round(2)}))
        fig, ax = plt.subplots(figsize=(6.5,4))
        conteos.plot.bar(ax=ax)
        ax.set_title('Distribución de niveles de participación')
        ax.set_xlabel('Nivel de participación'); ax.set_ylabel('Jugadores')
        ax.tick_params(axis='x', rotation=0)
        fig.tight_layout()
        fig.savefig(FIGURAS / '05_objetivo_participacion.png', dpi=150)
        plt.show()
        columnas_grupo = [c for c in ['PlayTimeHours','SessionsPerWeek','PlayerLevel','AvgSessionDurationMinutes'] if c in datos.columns]
        if columnas_grupo:
            print('Promedio de variables por nivel de participación:')
            display(datos.groupby(OBJETIVO)[columnas_grupo].mean().round(2).reindex([o for o in ORDEN if o in datos[OBJETIVO].unique()]))
    else:
        print('No aparece EngagementLevel; verifiquen el CSV y consulten al docente si se usará otra etiqueta.')

### 8.1 ¿Qué variables separan mejor los niveles de participación?
Comparamos la distribución de cada variable numérica según `EngagementLevel`. Si las cajas de Low, Medium y High casi se empalman, esa variable aportará poco al clasificador. Esto es exploratorio: **no entrena ningún modelo**.

In [ ]:
if datos is not None and OBJETIVO in datos.columns:
    pred = [c for c in ['PlayTimeHours','SessionsPerWeek','AvgSessionDurationMinutes','PlayerLevel','AchievementsUnlocked','Age'] if c in datos.columns]
    orden_presente = [o for o in ORDEN if o in datos[OBJETIVO].unique()]
    if pred:
        n = len(pred)
        filas_g = (n + 2) // 3
        fig, ejes = plt.subplots(filas_g, 3, figsize=(14, 4*filas_g), squeeze=False)
        for eje, col in zip(ejes.flat, pred):
            sns.boxplot(data=datos, x=OBJETIVO, y=col, order=orden_presente, ax=eje)
            eje.set_title(col + ' por nivel de participación')
            eje.set_xlabel('Nivel de participación')
        for eje in list(ejes.flat)[n:]: eje.axis('off')
        fig.tight_layout()
        fig.savefig(FIGURAS / '05b_predictores_vs_objetivo.png', dpi=150)
        plt.show()
    if 'InGamePurchases' in datos.columns:
        print('Proporción de jugadores con compras dentro del juego, por nivel de participación:')
        display(datos.groupby(OBJETIVO)['InGamePurchases'].mean().round(3).to_frame('proporcion_con_compras'))
    for cat in [c for c in ['GameGenre','GameDifficulty'] if c in datos.columns]:
        print('Distribución de', OBJETIVO, 'dentro de', cat, '(% por fila):')
        display((pd.crosstab(datos[cat], datos[OBJETIVO], normalize='index')*100).round(1))

## 9. Revisar imágenes FER-2013 por clase
Estructura esperada en `data/raw/fer2013/` (`train/` y `test/`, una carpeta por emoción). Se usan cinco emociones: enojado, feliz, neutral, triste y sorprendido.

In [ ]:
CLASES = {'angry':'Enojado', 'happy':'Feliz', 'neutral':'Neutral', 'sad':'Triste', 'surprise':'Sorprendido'}
EXTS = {'.jpg','.jpeg','.png','.bmp'}
filas_img = []
archivos_por_clase = {}
if DATOS.exists():
    splits = [p for p in DATOS.rglob('*') if p.is_dir() and p.name.lower() in {'train','test','val','validation'}]
    for carpeta_split in splits:
        for carpeta_clase in carpeta_split.iterdir():
            if not carpeta_clase.is_dir(): continue
            clave = carpeta_clase.name.lower()
            if clave in CLASES:
                ejemplos = [p for p in carpeta_clase.iterdir() if p.is_file() and p.suffix.lower() in EXTS]
                if not ejemplos: continue
                split = carpeta_split.name.lower()
                filas_img.append({'split':split,'clase':CLASES[clave],'imagenes':len(ejemplos)})
                archivos_por_clase.setdefault(CLASES[clave],[]).extend(ejemplos)

tabla_imagenes = pd.DataFrame(filas_img)
if tabla_imagenes.empty:
    print('Aún no hay carpetas de imágenes FER-2013 en data/raw/.')
    print('Descarga y descomprime: https://www.kaggle.com/datasets/msambare/fer2013')
else:
    print('Conteo de imágenes por conjunto y emoción:')
    display(tabla_imagenes.pivot_table(index='clase', columns='split',values='imagenes',aggfunc='sum',fill_value=0))
    print('Total de imágenes en las cinco emociones:', int(tabla_imagenes['imagenes'].sum()))

### 9.1 Verificación de formato de las imágenes
La CNN usa imágenes de 48 × 48 píxeles en escala de grises; se revisa una muestra por clase.

In [ ]:
if archivos_por_clase:
    rng = np.random.default_rng(42)
    formatos = []
    for clase, lista in archivos_por_clase.items():
        muestra = rng.choice(len(lista), size=min(100, len(lista)), replace=False)
        for i in muestra:
            with Image.open(lista[i]) as img:
                formatos.append({'clase': clase, 'tamano': img.size, 'modo': img.mode})
    df_fmt = pd.DataFrame(formatos)
    display(df_fmt.groupby(['tamano','modo']).size().to_frame('imagenes_en_muestra'))
    ok = (df_fmt['tamano'] == (48, 48)).all()
    print('Todas las imágenes de la muestra miden 48x48:' , ok)

## 10. ¿Hay desbalance entre las emociones?
Graficamos el número de fotografías por emoción, y comparamos la clase con más ejemplos contra la de menos ejemplos. La diferencia puede requerir ponderar las clases durante el entrenamiento.

In [ ]:
if not tabla_imagenes.empty:
    conteo_emociones = tabla_imagenes.groupby('clase')['imagenes'].sum().sort_values(ascending=False)
    display(pd.DataFrame({'imagenes':conteo_emociones,'porcentaje':(conteo_emociones/conteo_emociones.sum()*100).round(2)}))
    fig, ax = plt.subplots(figsize=(8,4))
    conteo_emociones.plot.bar(ax=ax)
    ax.set_title('Cantidad de imágenes FER-2013 por emoción (5 clases)')
    ax.set_xlabel('Emoción'); ax.set_ylabel('Número de imágenes')
    ax.tick_params(axis='x', rotation=0)
    fig.tight_layout()
    fig.savefig(FIGURAS / '06_balance_emociones.png', dpi=150)
    plt.show()
    razon = conteo_emociones.max()/conteo_emociones.min()
    print(f'Proporción clase mayor / clase menor: {razon:.2f}')
    print('Interpretación:', 'hay diferencia notable entre clases; revisar ponderación o aumento de datos.' if razon >= 1.5 else 'las clases están relativamente cercanas en tamaño.')

### 10.1 Pesos de clase
Pesos inversos a la frecuencia de cada clase, calculados solo con el conjunto de entrenamiento.

In [ ]:
if not tabla_imagenes.empty:
    split_train = 'train' if 'train' in tabla_imagenes['split'].values else tabla_imagenes['split'].iloc[0]
    c_train = tabla_imagenes[tabla_imagenes['split'] == split_train].groupby('clase')['imagenes'].sum()
    pesos = (c_train.sum() / (len(c_train) * c_train)).round(3)
    display(pd.DataFrame({'imagenes_train': c_train, 'peso_sugerido': pesos}))
    print('Fórmula: peso = total / (n_clases x imágenes de la clase). Un peso > 1 da más importancia a la clase minoritaria.')

## 11. Ejemplos de fotografías por emoción
Se mostrará un ejemplo real de cada clase disponible. **No se crean imágenes artificiales** en este análisis.

In [ ]:
if archivos_por_clase:
    clases_disponibles = [c for c in CLASES.values() if c in archivos_por_clase]
    fig, ejes = plt.subplots(1, len(clases_disponibles), figsize=(2.7*len(clases_disponibles),3), squeeze=False)
    for eje, clase in zip(ejes.flat, clases_disponibles):
        archivo = sorted(archivos_por_clase[clase])[0]
        with Image.open(archivo) as img:
            eje.imshow(img.convert('L'), cmap='gray')
        eje.set_title(clase); eje.axis('off')
    fig.suptitle('Ejemplo por clase - FER-2013')
    fig.tight_layout()
    fig.savefig(FIGURAS / '07_ejemplos_emociones.png', dpi=150)
    plt.show()

## 12. Interpretación de resultados

1. **Distribuciones:**
2. **Faltantes:**
3. **Correlaciones:**
4. **Atípicos:**
5. **Participación (`Low`, `Medium`, `High`):**
6. **Imágenes y balance de clases:**

**Variable objetivo:** `EngagementLevel`. El abandono se considera solo una hipótesis de riesgo, porque los datos tabulares no incluyen una etiqueta real de abandono.

## 13. Objetivos y métricas para las siguientes semanas

| Módulo | Qué se quiere hacer | Cómo evaluarlo |
|---|---|---|
| Supervisado | Clasificar `EngagementLevel` | F1 macro, AUC multiclase OvR y matriz de confusión; división 80/20, CV de 5 particiones |
| No supervisado | Segmentar estilos de juego | Codo (inercia) y coeficiente de silueta; PCA para visualizar |
| CNN | Reconocer 5 emociones faciales | Accuracy, F1 macro y matriz de confusión (referencia: 55-65 % de exactitud es razonable en FER-2013) |
| Refuerzo | Aprender a resolver FrozenLake 4×4 | Tasa de éxito en 100 episodios y recompensa media (con/sin deslizamiento) |